## TabNet: Attentive Interpretable Tabular Learning

"We propose a novel high-performance and interpretable canonical deep tabular data learning architecture, TabNet. TabNet uses sequential attention to choose which features to reason from at each decision step, enabling interpretability and more efficient learning as the learning capacity is used for the most salient features. We demonstrate that TabNet outperforms other neural network and decision tree variants on a wide range of non-performance-saturated tabular datasets and yields interpretable feature attributions plus insights into the global model behavior. Finally, for the first time to our knowledge, we demonstrate self-supervised learning for tabular data, significantly improving performance with unsupervised representation learning when unlabeled data is abundant." 

[TabNet Paper](https://arxiv.org/abs/1908.07442)


## Install PyTorch TabNet

In [1]:
!pip install pytorch-tabnet

You should consider upgrading via the 'C:\Users\venka\Downloads\Zelestra Pytorch tabular\torch_tabular\Scripts\python.exe -m pip install --upgrade pip' command.


## Parameters

In [2]:
# To ignore warinings
import warnings
warnings.filterwarnings('ignore')

In [3]:
NUM_FOLDS = 5
SEED = 42

## TabNet Parameters

In [4]:
## TabNet Parameters
MAX_EPOCH = 300
N_D = 6 
N_A = 6 
N_STEPS = 2
GAMMA = 1.1
LAMBDA_SPARSE = 1e-4
OPT_LR = 1e-5
OPT_WEIGHT_DECAY = 1e-5
OPT_MOMENTUM = 0.9
MASK_TYPE = "entmax"
SCHEDULER_MIN_LR = 1e-6
SCHEDULER_FACTOR = 0.9
DEVICE_NAME = "cuda"

BATCH_SIZE = 32

## Imports Libs

In [5]:
import torch
from torch import nn
from pytorch_tabnet.tab_model import TabNetRegressor
import optuna

from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd 

import os
import random
import sys
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
seed_everything(SEED)

## Import Data

In [6]:
import pandas as pd

train = pd.read_csv("C:/Users/venka/Downloads/1a513221-3-dataset/dataset/train.csv")

train = train.drop(columns=['id'])
test=pd.read_csv("C:/Users/venka/Downloads/1a513221-3-dataset/dataset/test.csv").drop(columns=['id'])

In [7]:
train.dtypes

temperature           float64
irradiance            float64
humidity               object
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed             object
pressure               object
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [8]:
columns = train.columns.to_list()
features = columns[:-1]
target = columns[-1]

print(f'Features: {features}')
print(f'Target: {target}')

Features: ['temperature', 'irradiance', 'humidity', 'panel_age', 'maintenance_count', 'soiling_ratio', 'voltage', 'current', 'module_temperature', 'cloud_coverage', 'wind_speed', 'pressure', 'string_id', 'error_code', 'installation_type']
Target: efficiency


In [9]:
obj_num_cols=train.select_dtypes(include='object').columns[:-3]

In [10]:
obj_num_cols

Index(['humidity', 'wind_speed', 'pressure'], dtype='object')

In [11]:
for col in obj_num_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')

In [12]:
train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [13]:
for col in obj_num_cols:
    test[col] = pd.to_numeric(test[col], errors='coerce')

In [14]:
X_train = train[features]
y_train = train[target]

In [15]:
X_train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [16]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')


pt.fit(X_train[list(X_train.select_dtypes(include='float64').columns)])
X_train_pt = pt.transform(X_train[list(X_train.select_dtypes(include='float64').columns)])

print(f'Shape of X_train_pt: {X_train_pt.shape}')

Shape of X_train_pt: (20000, 12)


In [17]:
X_train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [18]:
pt.fit(test[list(test.select_dtypes(include='float64').columns)])
test_pt = pt.transform(test[list(test.select_dtypes(include='float64').columns)])

In [19]:
test_pt=pd.DataFrame(test_pt,columns=list(test.select_dtypes(include='float64').columns))
X_train_pt = pd.DataFrame(X_train_pt, columns=list(X_train.select_dtypes(include='float64').columns))

In [20]:
X_train_pt[list(X_train.select_dtypes(include='object').columns)]=X_train[list(X_train.select_dtypes(include='object').columns)]

In [21]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [22]:
test_pt[list(test.select_dtypes(include='object').columns)]=test[list(test.select_dtypes(include='object').columns)]

In [23]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [24]:
from sklearn.preprocessing import LabelEncoder

# Sample: list of columns to encode
cols_to_encode = list(X_train_pt.select_dtypes(include='object').columns)

le = LabelEncoder()
for col in cols_to_encode:
    # Convert all values to string to avoid mixed types (also handles NaN as 'nan')
    X_train_pt[col] = X_train_pt[col].astype(str)
    X_train_pt[col] = le.fit_transform(X_train_pt[col])

In [25]:
for col in cols_to_encode:
    test_pt[col] = test_pt[col].astype(str)
    test_pt[col] = le.fit_transform(test_pt[col])

In [26]:
target = train.pop("efficiency")
target = target.values

## Label Encoding

In [27]:
columns = test.columns

## Create TabNet Params Dictionary

In [28]:
def Objective(trial):
    mask_type = trial.suggest_categorical("mask_type", ["entmax", "sparsemax"])
    n_da = trial.suggest_int("n_da", 56, 64, step=4)
    n_steps = trial.suggest_int("n_steps", 1, 3, step=1)
    gamma = trial.suggest_float("gamma", 1., 1.4, step=0.2)
    n_shared = trial.suggest_int("n_shared", 1, 3)
    lambda_sparse = trial.suggest_float("lambda_sparse", 1e-6, 1e-3, log=True)
    tabnet_params = dict(n_d=n_da, n_a=n_da, n_steps=n_steps, gamma=gamma,
                     lambda_sparse=lambda_sparse, optimizer_fn=torch.optim.Adam,
                     optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
                     mask_type=mask_type, n_shared=n_shared,
                     scheduler_params=dict(mode="min",
                                           patience=trial.suggest_int("patienceScheduler",low=3,high=10), # changing sheduler patience to be lower than early stopping patience 
                                           min_lr=1e-5,
                                           factor=0.5,),
                     scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
                     verbose=0,
                     ) #early stopping
    kf = KFold(n_splits=5, random_state=42, shuffle=True)
    CV_score_array    =[]
    train_oof = np.zeros((len(train)))
    
    for f, (train_ind, val_ind) in tqdm(enumerate(kf.split(X_train_pt, target))):
        print(f'Fold {f}')
        train_df, val_df = X_train_pt.iloc[train_ind][columns], X_train_pt.iloc[val_ind][columns]
    
        train_target, val_target = target[train_ind], target[val_ind]
    
        print(X_train_pt.shape, train_target.shape)
        print(val_df.shape, val_target.shape)
    
        train_target=train_target.reshape(-1,1)
        val_target=val_target.reshape(-1,1)
    
        train_df      = train_df.to_numpy()
        train_target      = train_target.reshape(-1, 1)
    
        val_df = val_df.to_numpy()
        val_target = val_target.reshape(-1, 1)
    
        model = TabNetRegressor(**tabnet_params)
    
        model.fit(X_train=train_df,
                  y_train=train_target,
                  eval_set=[(val_df, val_target)],
                  eval_name = ["val"],
                  eval_metric = ['rmse'],#["logits_ll"],
                  patience=trial.suggest_int("patience",low=15,high=30), max_epochs=trial.suggest_int('epochs', 1, 100),
                  batch_size=BATCH_SIZE)
    
        CV_score_array.append(model.best_cost)
    avg = np.mean(CV_score_array)
    return avg

In [29]:
import numpy as np

X_train_pt.replace([np.inf, -np.inf], np.nan, inplace=True)
test_pt.replace([np.inf, -np.inf], np.nan, inplace=True)
from sklearn.impute import SimpleImputer

numeric_cols = X_train_pt.select_dtypes(include=[np.number]).columns

# Create the imputer (you can use 'mean', 'median', or 'most_frequent')
imputer = SimpleImputer(strategy='mean')

# Fit and transform only on numeric columns
X_train_pt[numeric_cols] = imputer.fit_transform(X_train_pt[numeric_cols])
numeric_cols = test_pt.select_dtypes(include=[np.number]).columns
test_pt[numeric_cols] = imputer.fit_transform(test_pt[numeric_cols])

In [ ]:
study = optuna.create_study(direction="minimize", study_name='TabNet Zelestra Optimization',storage="sqlite:///db.sqlite3")
study.optimize(Objective, timeout=60*5) #5 hours

[I 2025-06-02 10:25:43,585] Using an existing study with name 'TabNet Zelestra Optimization' instead of creating a new one.


0it [00:00, ?it/s]

Fold 0
(20000, 15) (16000,)
(4000, 15) (4000,)


UnboundLocalError: local variable 'frozen_trial' referenced before assignment

In [ ]:
TabNet_params = study.best_params

In [ ]:
print(TabNet_params)

In [ ]:
import json

with open("hyper.json","w") as f:
    json.dump(TabNet_params,f)